# Exp8.0.1 — Frozen L1/L2 Fusion Probe

This notebook is aggregation-only. It reads finalized Exp8.0.1 CSV/JSON artifacts and does not retrain the SNN or refit probes.

Primary question: does the frozen Exp8.0 `234x234` backbone contain complementary information across L1 and L2, especially `L1 whole + L2 Fixed250`?

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
while repo.name != 'writingRing' and repo.parent != repo:
    repo = repo.parent
base = repo / 'notebooks' / 'artifacts' / 'experiment_8_0_1_l1_l2_fusion_probe' / 'l1_l2_fusion_probe_v1'
manifest = json.loads((base / 'manifest.json').read_text())
probe_runs = pd.read_csv(base / 'probe_runs.csv')
probe_summary = pd.read_csv(base / 'probe_summary.csv')
fusion_runs = pd.read_csv(base / 'fusion_gain_runs.csv')
fusion_summary = pd.read_csv(base / 'fusion_gain_summary.csv')
overlap_runs = pd.read_csv(base / 'correctness_overlap_runs.csv')
overlap_summary = pd.read_csv(base / 'correctness_overlap_summary.csv')
coef_runs = pd.read_csv(base / 'coef_block_runs.csv')
coef_summary = pd.read_csv(base / 'coef_block_summary.csv')
manifest

## Main frozen-probe comparison

The important comparison is test BA and its paired-seed behavior, not training fit. The train-test gap is shown explicitly because Fixed250 fusion is high-dimensional.

In [ ]:
cols = [
    'feature', 'feature_dim_mean',
    'train_ba_mean', 'val_ba_mean', 'test_ba_mean', 'test_ba_std',
    'train_test_gap_mean', 'probe_C_mean',
]
display(probe_summary[cols].sort_values('test_ba_mean', ascending=False))

In [ ]:
plot = probe_summary.set_index('feature')
ax = plot['test_ba_mean'].plot.bar(yerr=plot['test_ba_std'], capsize=3, figsize=(11, 4))
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp8.0.1 frozen L1/L2 probe comparison')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Fusion gains

Positive values mean the fused feature beats the relevant best single-layer component on the same seed.

In [ ]:
display(fusion_runs)
display(fusion_summary.sort_values('mean', ascending=False))

## L1/L2 correctness overlap

`l1_only_correct` and `l2_only_correct` quantify error complementarity directly. `oracle_union_accuracy` is diagnostic only; it is not a deployable classifier.

In [ ]:
test_overlap = overlap_summary[overlap_summary['split'] == 'test'].copy()
display(test_overlap)

## Fusion coefficient blocks

Features are standardized before the probe. `coef_rms` is therefore useful for checking whether a fused probe uses both L1 and L2 rather than effectively ignoring one block.

In [ ]:
multi = coef_summary[coef_summary['feature'].isin([
    'l1_l2_whole', 'l1_l2_fixed250',
    'l1whole_l2fixed250', 'l1fixed250_l2whole'
])].copy()
display(multi[[
    'feature', 'block_index', 'layer', 'aggregation',
    'coef_rms_mean', 'coef_rms_std',
    'coef_norm_fraction_mean', 'coef_norm_fraction_std',
]])

## Interpretation checklist

- If `l1_l2_whole` or `l1_l2_fixed250` consistently beats the stronger corresponding single layer, L1 and L2 contain complementary information at that aggregation scale.
- If `l1whole_l2fixed250` is strongest and both coefficient blocks remain active, the data support an explicit multi-level path preserving L1 aggregate evidence while retaining L2 phase-aware evidence.
- If fusion improves train BA but not validation/test BA, prioritize invariance/augmentation rather than a larger skip architecture.
- If L1-only-correct is near zero, there is little evidence that a skip path is necessary even if raw dimensions increase.